# Micro-Hay spike fine-tuning 03

Controlled comparison between the converged GRU-MSE and the same model fine-tuned with a conservative rare-event objective. The global 61-state MSE never disappears; spike terms enter through a curriculum. No teacher state is used as input or feedback.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} exists but is not the project')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## Fine-tuning
The script auto-discovers saved Kaggle inputs when the dataset or baseline checkpoint is not in `/kaggle/working`. It writes `last` after every completed epoch and resumes automatically after interruptions.

In [ ]:
os.environ['HAY_FINETUNE_DATASET'] = '/kaggle/working/hay_micro_4c_event_enriched_v2.h5'
os.environ['HAY_FINETUNE_BASELINE'] = '/kaggle/working/hay_micro_event_aware_02/checkpoints/gru_mse.pt'
os.environ['HAY_FINETUNE_OUTPUT'] = '/kaggle/working/hay_micro_spike_finetune_03'
os.environ['HAY_FINETUNE_EPOCHS'] = '12'
os.environ['HAY_FINETUNE_WINDOWS_PER_EPOCH'] = '24'
os.environ['HAY_FINETUNE_FORCE_RESTART'] = '0'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_spike_finetune_03.py

In [ ]:
from IPython.display import Image, display
display(pd.read_csv('/kaggle/working/hay_micro_spike_finetune_03/comparison.csv').T)
display(Image('/kaggle/working/hay_micro_spike_finetune_03/soma_comparison.png'))

In [ ]:
from pathlib import Path
from shutil import make_archive
from IPython.display import FileLink, display
source = Path('/kaggle/working/hay_micro_spike_finetune_03')
zip_path = Path(make_archive('/kaggle/working/hay_micro_spike_finetune_03_complete', 'zip', root_dir=source.parent, base_dir=source.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))